In [1]:
import os
import json
import time
import requests
import numpy as np
import twelvelabs
from dotenv import load_dotenv
from twelvelabs.models.embed import SegmentEmbedding
from typing import List
import requests
import io
from PIL import Image, ImageOps
import pickle
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize
import pickle
load_dotenv()

True

In [2]:
available_results = os.listdir('contrastive_results/custom')


FileNotFoundError: [Errno 2] No such file or directory: 'contrastive_results/custom'

In [3]:
available_results

NameError: name 'available_results' is not defined

In [ ]:

all_results = []
val = 0.5

for i in range(0,5):
    for result in available_results:

        with open("/home/ubuntu/code/libby/pipeline/data/finetuned_may19.pkl", 'rb') as file:
            resnet_df = pickle.load(file)
        
        resnet_df = resnet_df[['video', 'frame', 'owl_label', 'finetuned_embedding']]

  

        with open(f"/home/ubuntu/code/contrastive_results/custom/{result}/representatives.pkl", 'rb') as file:

            defObjects = pickle.load(file)

            class_emb_matrix = list(defObjects['finetuned_embedding'])
            objs = list(defObjects['class'])

            def cosine_scores(input_emb, class_emb_matrix):
                input_tensor = F.normalize(torch.tensor(input_emb, dtype=torch.float32).unsqueeze(0), dim=1)
                class_tensor = F.normalize(torch.tensor(np.vstack(class_emb_matrix), dtype=torch.float32), dim=1)
                return F.cosine_similarity(input_tensor, class_tensor).numpy()



            def visual_prediction_from_text_filtered(row):
                visual_scores = cosine_scores(row['finetuned_embedding'], class_emb_matrix)
                best_idx = np.argmax(visual_scores)
                return pd.Series({
                    'visual_predicted_object': objs[best_idx],
                    'visual_max_score': visual_scores[best_idx],
                })

            visual_preds = resnet_df.apply(visual_prediction_from_text_filtered, axis=1)

            print(visual_preds)
            resnet_df[['visual_predicted_object', 'visual_max_score']] = visual_preds

            visual_threshold = val + 0.05*i # THIS WILL CHANGE -- NEED TO TEST FOR YOUR STUFF
            print('visual_threshold: ', visual_threshold)
            resnet_df['prediction'] = 1
            resnet_df['frame'] = resnet_df['frame'].apply(lambda row: int(row))
            print('resnet_df: ', resnet_df)
            # idx = filtered_df_copy.groupby(['video', 'visual_predicted_object'])['prediction'].idxmax()
            idx = resnet_df.groupby(['video', 'visual_predicted_object'])['visual_max_score'].idxmax()
            resnet_df = resnet_df.loc[idx].reset_index(drop=True)

            resnet_df['visual_predicted_object'] = resnet_df.apply(lambda row: row['visual_predicted_object'] if row['visual_max_score'] > visual_threshold else 'No Class', axis=1)
            resnet_df = resnet_df[resnet_df['visual_predicted_object'] != 'No Class']


            with open("/home/ubuntu/code/libby/pipeline/data/sourceTruth_jeremiah.pkl", 'rb') as file:
                final_SOT = pickle.load(file)
                
            final_SOT = final_SOT.rename(columns={'frame': 'second', 'second': 'frame'})
            final_SOT = final_SOT.groupby(['video', 'tag'])['actual'].max().reset_index()
            # final_SOT['frame'] = final_SOT['frame'].apply(lambda row: int(row))

            merged = pd.merge(resnet_df, final_SOT, left_on=['video', 'visual_predicted_object'], right_on=['video', 'tag'], how='right')


            def classify_answer(answer, pred):
                if answer == 0 and pred == 0:
                    return 'TN'
                elif answer == 1 and pred == 1:
                    return 'TP'
                elif answer == 1 and pred == 0:
                    return 'FN'
                return 'FP'
            merged['prediction'] = merged['prediction'].fillna(0)
            merged['answerClass'] =  merged.apply(lambda row: classify_answer(row['actual'], row['prediction']), axis = 1)

            counts = merged['answerClass'].value_counts()
            TP = counts.get('TP', 0)
            FP = counts.get('FP', 0)
            FN = counts.get('FN', 0)
            TN = counts.get('TN', 0)


        

            # Calculate metrics
            print('TP: ', TP)
            print('FP: ', FP)
            print('FN: ', FN)
            print('TN: ', TN)
            precision = TP / (TP + FP) if (TP + FP) > 0 else 0
            recall = TP / (TP + FN) if (TP + FN) > 0 else 0
            f1_score = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0



            result_summary = {
                'result': result,
                'TP': TP,
                'FP': FP,
                'FN': FN,
                'TN': TN,
                'precision': precision,
                'recall': recall,
                'f1_score': f1_score,
                'visual_threshold': visual_threshold,
            }
            all_results.append(result_summary)

        # Display
        # print(f'Precision: {precision:.4f}')
        # print(f'Recall:    {recall:.4f}')
        # print(f'F1 Score:  {f1_score:.4f}')

        # alx = merged.groupby(["tag","answerClass"]).size().reset_index().pivot(columns='answerClass', values=0, index = 'tag').fillna(0).reset_index().sort_values('TP')
        # alx['precision'] = alx['TP'] / (alx['TP'] + alx['FP']) 
        # alx['recall'] = alx['TP'] / (alx['TP'] + alx['FN']) 
        # alx['f-score'] = (alx['precision']*alx['recall']/(alx['precision'] + alx['recall']))*2
        
        # alx['Total Labels Across Videos'] = alx['TP'] + alx['FN']
        
        # alx.fillna(0).sort_values('f-score', ascending = False)

/tmp/ipykernel_1903/3030878656.py:8: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  resnet_df = pickle.load(file)


         visual_predicted_object  visual_max_score
0           TVA Prisoner Uniform          0.669258
1           TVA Prisoner Uniform          0.687412
2           TVA Prisoner Uniform          0.690408
3          Aligator Loki Plushie          0.659405
4                 Sylvie's Armor          0.665495
...                          ...               ...
18323  Sylvie's horned headpiece          0.659311
18324                  TimeSpear          0.682474
18325                 TVA Collar          0.587798
18326       TVA Prisoner Uniform          0.690308
18327                  TimeSpear          0.868422

[18328 rows x 2 columns]
visual_threshold:  0.5
resnet_df:                                                 video  frame  \
0        Scenes 001-020__314-3_20230815232058756.mp4    112   
1        Scenes 001-020__314-3_20230815232058756.mp4     58   
2        Scenes 001-020__314-3_20230815232058756.mp4    115   
3        Scenes 001-020__314-3_20230815232058756.mp4     83   
4        Sce

/tmp/ipykernel_1903/3030878656.py:55: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  final_SOT = pickle.load(file)
/tmp/ipykernel_1903/3030878656.py:8: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the ca

         visual_predicted_object  visual_max_score
0           TVA Prisoner Uniform          0.613029
1           TVA Prisoner Uniform          0.630115
2           TVA Prisoner Uniform          0.637875
3          Aligator Loki Plushie          0.608685
4                 Sylvie's Armor          0.604606
...                          ...               ...
18323  Sylvie's horned headpiece          0.606356
18324                  TimeSpear          0.627274
18325                 TVA Collar          0.535224
18326       TVA Prisoner Uniform          0.633449
18327                  TimeSpear          0.831886

[18328 rows x 2 columns]
visual_threshold:  0.5
resnet_df:                                                 video  frame  \
0        Scenes 001-020__314-3_20230815232058756.mp4    112   
1        Scenes 001-020__314-3_20230815232058756.mp4     58   
2        Scenes 001-020__314-3_20230815232058756.mp4    115   
3        Scenes 001-020__314-3_20230815232058756.mp4     83   
4        Sce

/tmp/ipykernel_1903/3030878656.py:55: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  final_SOT = pickle.load(file)
/tmp/ipykernel_1903/3030878656.py:8: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the ca

...

/tmp/ipykernel_1903/3030878656.py:55: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  final_SOT = pickle.load(file)
/tmp/ipykernel_1903/3030878656.py:8: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the ca

         visual_predicted_object  visual_max_score
0           TVA Prisoner Uniform          0.657228
1           TVA Prisoner Uniform          0.672014
2           TVA Prisoner Uniform          0.679791
3          Aligator Loki Plushie          0.650488
4                 Sylvie's Armor          0.672516
...                          ...               ...
18323  Sylvie's horned headpiece          0.668357
18324                  TimeSpear          0.654693
18325                 TVA Collar          0.598563
18326       TVA Prisoner Uniform          0.673821
18327                  TimeSpear          0.853381

[18328 rows x 2 columns]
visual_threshold:  0.9
resnet_df:                                                 video  frame  \
0        Scenes 001-020__314-3_20230815232058756.mp4    112   
1        Scenes 001-020__314-3_20230815232058756.mp4     58   
2        Scenes 001-020__314-3_20230815232058756.mp4    115   
3        Scenes 001-020__314-3_20230815232058756.mp4     83   
4        Sce

/tmp/ipykernel_1903/3030878656.py:55: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  final_SOT = pickle.load(file)
/tmp/ipykernel_1903/3030878656.py:8: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the ca

         visual_predicted_object  visual_max_score
0           TVA Prisoner Uniform          0.685713
1           TVA Prisoner Uniform          0.700763
2           TVA Prisoner Uniform          0.706460
3          Aligator Loki Plushie          0.680373
4                 Sylvie's Armor          0.708396
...                          ...               ...
18323  Sylvie's horned headpiece          0.699779
18324                  TimeSpear          0.687350
18325                 TVA Collar          0.628280
18326       TVA Prisoner Uniform          0.702694
18327                  TimeSpear          0.874758

[18328 rows x 2 columns]
visual_threshold:  0.9
resnet_df:                                                 video  frame  \
0        Scenes 001-020__314-3_20230815232058756.mp4    112   
1        Scenes 001-020__314-3_20230815232058756.mp4     58   
2        Scenes 001-020__314-3_20230815232058756.mp4    115   
3        Scenes 001-020__314-3_20230815232058756.mp4     83   
4        Sce

/tmp/ipykernel_1903/3030878656.py:55: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  final_SOT = pickle.load(file)


In [77]:
all_results = pd.DataFrame(all_results)

In [78]:
sort_cols = [ 'f1_score']

In [79]:
sorted_res = all_results.sort_values(by=sort_cols, ascending=False)

In [80]:
sorted_res

,result,TP,FP,FN,TN,precision,recall,f1_score,visual_threshold
14,quick_test_balanced,68,62,68,1620,0.523077,0.500000,0.511278,0.8
15,first_pass,76,88,60,1594,0.463415,0.558824,0.506667,0.8
12,very_agg,66,65,70,1617,0.503817,0.485294,0.494382,0.8
13,quick_test_agg,54,31,82,1651,0.635294,0.397059,0.488688,0.8
9,quick_test_agg,87,156,49,1526,0.358025,0.639706,0.459103,0.7
19,first_pass,37,6,99,1676,0.860465,0.272059,0.413408,0.9
10,quick_test_balanced,105,272,31,1410,0.278515,0.772059,0.409357,0.7
8,very_agg,105,287,31,1395,0.267857,0.772059,0.397727,0.7
18,quick_test_balanced,34,2,102,1680,0.944444,0.250000,0.395349,0.9
16,very_agg,34,2,102,1680,0.944444,0.250000,0.395349,0.9
